# Cassini CDA Dust Analyzer: 3-OH Inference on Unlabelled Spectra

This notebook sends unlabelled CDA spectra to Gemini in Vertex AI batch jobs of up to 1,000 spectra. Each spectrum is labelled **3-OH** or **non-3-OH**, using the few-shot pool written by `15_3oh_few_shot_study.ipynb`. It mirrors `Notebooks/XX_peaks_focused_inference.ipynb`.

**Before running:** run notebook 15 at least through Step 7, so that `study_results_3oh/few_shot_pool_3oh.json` exists, and check its metrics.

### Features
1. **Same exemplars, prompts and model as the tested study.** All of them are read from the pool file. The preprocessing and plot settings are checked against it.
2. **Choice of unlabelled spectra** (`INFERENCE_SCOPE`):
   - `catalogue_type3` (default): spectra that the catalogue lists only as generic Type `3`, never sub-classified. That is 492 in the Q_I window, minus the one used as an exemplar.
   - `all`: every unlabelled spectrum in the Q_I window, about 205,000, i.e. about 205 batch jobs. Use `MAX_BATCHES` to limit a run.
   - `sclk_file`: the SCLKs in `SCLK_FILE`.
3. **Resumable state.** Spectra with a usable answer are recorded in `processed_sclks.json` and skipped on the next run. Spectra without one are retried.
4. **Dry run.** `DRY_RUN = True` writes the first request file without submitting it, so you can inspect its size and content.

### Outputs (`Notebooks/3oh_classification/inference_results_3oh/`)
- `predictions_3oh.csv`: every answered spectrum with sclk, event_id, qi_ampl, catalogue class (if any), predicted label, explanation and batch job
- `processed_sclks.json`: SCLKs with a usable answer
- `candidates_3oh.csv`: the spectra labelled 3-OH
- `candidate_plots/<sclk>.png`: the image Gemini saw, for every 3-OH candidate

## Step 1: Imports and Configuration

In [1]:
import os
import sys
import json
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv

import warnings
import logging
warnings.filterwarnings("ignore")
logging.getLogger("google_genai").setLevel(logging.WARNING)

# Repository root = the folder with pyproject.toml and the cda_dust_agent package, wherever the kernel was started.
# (Notebooks/cda_dust_agent/ is only a data folder, so the package's __init__.py is required.)
def is_repo_root(path):
    return (os.path.isfile(os.path.join(path, "pyproject.toml"))
            and os.path.isfile(os.path.join(path, "cda_dust_agent", "__init__.py")))

project_root = os.path.abspath(os.getcwd())
while not is_repo_root(project_root):
    parent = os.path.dirname(project_root)
    if parent == project_root:
        raise RuntimeError("Repository root (pyproject.toml + cda_dust_agent/) not found; start Jupyter inside the repository.")
    project_root = parent
study_dir = os.path.join(project_root, "Notebooks", "3oh_classification")
print(f"Repository root: {project_root}")

if not load_dotenv(dotenv_path=os.path.join(project_root, ".env")):
    print("Warning: no .env loaded; the defaults in cda_dust_agent/config.py are used for project and bucket.")
sys.path.extend([project_root, study_dir])

from cda_dust_agent.config import Config
import cda_3oh_common as c3

Repository root: /workspaces/cda-dust-analyzer


In [2]:
POOL_PATH = os.path.join(study_dir, "study_results_3oh", "few_shot_pool_3oh.json")
STATE_DIR = os.path.join(study_dir, "inference_results_3oh")
CATALOGUE_CSV = os.path.join(project_root, "Notebooks", "cda_class_list.csv")

# Which unlabelled spectra (HF inference split) to send:
#   "catalogue_type3": spectra the catalogue lists only as generic Type "3" (never sub-classified)
#   "all":             every unlabelled spectrum in the Q_I window (~205k spectra, ~205 batch jobs)
#   "sclk_file":       the SCLKs in SCLK_FILE (one SCLK per line, or a CSV with an 'sclk' column)
INFERENCE_SCOPE = "all"
SCLK_FILE = None
SCLK_RANGE = (None, None)  # optional (min, max) SCLK limits
APPLY_QI_WINDOW = True     # same 10 fC - 2 pC window as the study

BATCH_SIZE = 1000
MAX_BATCHES = 30         # None = run until the queue is empty
DRY_RUN = False            # True = write the first request file, then stop without submitting
SAVE_CANDIDATE_PLOTS = True

configs = Config()
os.makedirs(STATE_DIR, exist_ok=True)
print(f"Project: {configs.agent_settings.project_id} | bucket: {configs.agent_settings.bucket_name}")
print(f"State and results: {STATE_DIR}")

Project: turan-genai-bb | bucket: turansgenaibb
State and results: /workspaces/cda-dust-analyzer/Notebooks/3oh_classification/inference_results_3oh


## Step 2: Load the Few-Shot Pool from Notebook 15

In [3]:
with open(POOL_PATH) as f:
    pool = json.load(f)
if pool["time_base"] != c3.time_base_settings():
    raise ValueError(f"The pool was built with other preprocessing/plot settings:\n{pool['time_base']}\n"
                     f"vs. now:\n{c3.time_base_settings()}\nRerun notebook 15.")
if pool["model"] != configs.agent_settings.model:
    print(f"Note: config model is {configs.agent_settings.model}; using the pool's model {pool['model']} (the one that was tested).")

exemplar_sclks = {ex["sclk"] for ex in pool["exemplars"]}
label_counts = pd.Series([ex["label"] for ex in pool["exemplars"]]).value_counts().to_dict()
print(f"Few-shot pool from {pool['created']}: {len(pool['exemplars'])} exemplars {label_counts}, model {pool['model']}")
print("\nSystem instruction:\n" + pool["system_instruction"])

Few-shot pool from 2026-10-19T14:08:32: 21 exemplars {'non-3-OH': 12, '3-OH': 9}, model gemini-3.5-flash

System instruction:
You are an expert cosmic dust spectroscopist analysing time-of-flight (TOF) mass spectra recorded by the Cassini Cosmic Dust Analyzer (CDA).
Your task is to label each spectrum as '3-OH' or 'non-3-OH'.

How to read the plots:
- The x-axis is the ion time of flight in microseconds (0 to 6.4 µs; the detector is sampled at 100 MHz, i.e. 10 ns per sample). The y-axis is the log-scaled amplitude, normalised to 0-1.
- The flight time grows with the square root of the ion mass: t = t0 + 0.473 µs x sqrt(m/z). The offset t0 differs from spectrum to spectrum, so a given ion can appear several tenths of a µs earlier or later. Identify peaks by their time distance from the Na+ peak, not by their absolute position. Allow about ±0.1 µs for every time distance.
- Ignore the vertical level of the baseline and any baseline shift.

Class '3-OH' (salt-rich Type 3 ice grain contain

## Step 3: Select the Unlabelled Spectra

In [4]:
inf_path = c3.hf_path(c3.HF_INF_FILE)
catalogue = c3.load_catalogue(CATALOGUE_CSV)

meta = c3.read_metadata(inf_path)
print(f"Loaded metadata of {len(meta)} unlabelled spectra.")
meta = meta[meta["spectrum_length"] == c3.SPECTRUM_LENGTH]
if APPLY_QI_WINDOW:
    meta = c3.in_qi_window(meta)
    print(f"{len(meta)} in the Q_I window (10 fC - 2 pC).")
meta = meta.merge(catalogue[["sclk", "catalogue_class"]], on="sclk", how="left")

if INFERENCE_SCOPE == "catalogue_type3":
    meta = meta[meta["catalogue_class"] == "3"]
elif INFERENCE_SCOPE == "sclk_file":
    if not SCLK_FILE:
        raise ValueError("Set SCLK_FILE for INFERENCE_SCOPE = 'sclk_file'.")
    if SCLK_FILE.endswith(".csv"):
        listed = pd.read_csv(SCLK_FILE)["sclk"]
    else:
        with open(SCLK_FILE) as f:
            listed = pd.Series([line.strip() for line in f if line.strip()])
    meta = meta[meta["sclk"].isin(pd.to_numeric(listed, errors="coerce").dropna().astype("int64"))]
elif INFERENCE_SCOPE != "all":
    raise ValueError(f"Unknown INFERENCE_SCOPE: {INFERENCE_SCOPE}")

sclk_min, sclk_max = SCLK_RANGE
if sclk_min is not None:
    meta = meta[meta["sclk"] >= sclk_min]
if sclk_max is not None:
    meta = meta[meta["sclk"] <= sclk_max]
meta = meta[~meta["sclk"].isin(exemplar_sclks)].sort_values("sclk").reset_index(drop=True)
print(f"Scope '{INFERENCE_SCOPE}': {len(meta)} spectra (few-shot exemplars excluded).")
if meta["catalogue_class"].notna().any():
    print("Catalogue labels among them:", meta["catalogue_class"].value_counts().to_dict())

2026-10-19 15:44:29,630 - [httpx] - INFO - HTTP Request: HEAD https://huggingface.co/datasets/CosmicDustGroup/cassini-cda-spectra/resolve/main/data/lvl2/cda_qm_spectra_pre2008277_inf_lvl2.parquet "HTTP/1.1 302 Found"


Loaded metadata of 756070 unlabelled spectra.
204936 in the Q_I window (10 fC - 2 pC).
Scope 'all': 204929 spectra (few-shot exemplars excluded).
Catalogue labels among them: {'3': 491, 'Stream': 91, 'HMOC': 71, '3-P': 23, 'Test pulses': 8}


## Step 4: Load Progress and the Spectra of This Run

Already processed SCLKs are skipped. The raw spectra of this run's queue are read once. The inference file is a single 600 MB row group, so it is streamed in chunks.

In [5]:
processed_path = os.path.join(STATE_DIR, "processed_sclks.json")
predictions_path = os.path.join(STATE_DIR, "predictions_3oh.csv")
plots_dir = os.path.join(STATE_DIR, "candidate_plots")

processed = set()
if os.path.exists(processed_path):
    with open(processed_path) as f:
        processed = set(json.load(f))

queue = meta[~meta["sclk"].isin(processed)].reset_index(drop=True)
n_batches = int(np.ceil(len(queue) / BATCH_SIZE))
if MAX_BATCHES is not None:
    n_batches = min(n_batches, MAX_BATCHES)

request_mb = (sum(len(ex["image_base64"]) for ex in pool["exemplars"]) + 60_000) / 1e6
print(f"Already processed: {len(processed)}. Queue: {len(queue)} spectra -> {n_batches} batch job(s) in this run.")
print(f"Each request carries {len(pool['exemplars'])} exemplar images (~{request_mb:.1f} MB), "
      f"i.e. ~{request_mb * min(BATCH_SIZE, len(queue)):.0f} MB per request file.")

run_sclks = queue["sclk"].iloc[: n_batches * BATCH_SIZE]
raw_spectra = c3.read_spectra(inf_path, run_sclks).drop_duplicates("sclk").set_index("sclk")["spectrum"]
print(f"Loaded {len(raw_spectra)} raw spectra.")

Already processed: 0. Queue: 204929 spectra -> 30 batch job(s) in this run.
Each request carries 21 exemplar images (~1.3 MB), i.e. ~1321 MB per request file.
Loaded 29997 raw spectra.


## Step 5: Run the Batch Jobs

In [6]:
for b in range(n_batches):
    batch_meta = queue.iloc[b * BATCH_SIZE:(b + 1) * BATCH_SIZE].copy()
    print(f"\n--- Batch {b + 1}/{n_batches}: {len(batch_meta)} spectra "
          f"(SCLK {batch_meta['sclk'].min()} - {batch_meta['sclk'].max()}) ---")
    batch_meta["spectrum"] = list(raw_spectra.loc[batch_meta["sclk"]])
    batch_df = c3.preprocess(batch_meta)

    request_path = os.path.join(STATE_DIR, "batch_requests.jsonl")
    c3.write_request_file(request_path, batch_df, pool)
    if DRY_RUN:
        print(f"DRY_RUN: request file kept at {request_path}; nothing was submitted.")
        break

    job_status = c3.run_batch_job(configs.agent_settings.project_id, configs.agent_settings.bucket_name,
                                  pool["model"], request_path, display_prefix="cda-3oh-inf")
    predictions = pd.DataFrame(
        list(c3.iter_batch_predictions(configs.agent_settings.project_id, configs.agent_settings.bucket_name, job_status)),
        columns=["sclk", "predicted_label", "explanation", "error"],
    )
    answered = (predictions[predictions["predicted_label"].isin(c3.BINARY_LABELS)]
                .dropna(subset=["sclk"]).astype({"sclk": "int64"}).drop_duplicates("sclk"))
    answered = (batch_meta[["sclk", "event_id", "qi_ampl", "catalogue_class"]]
                .merge(answered[["sclk", "predicted_label", "explanation"]], on="sclk", how="inner"))
    answered["batch_job"] = job_status.get("name")
    answered["classified_at"] = datetime.now().isoformat(timespec="seconds")

    answered.to_csv(predictions_path, mode="a", header=not os.path.exists(predictions_path), index=False)
    processed.update(int(s) for s in answered["sclk"])
    with open(processed_path, "w") as f:
        json.dump(sorted(processed), f)

    if SAVE_CANDIDATE_PLOTS:
        os.makedirs(plots_dir, exist_ok=True)
        spectrum_by_sclk = batch_df.set_index("sclk")["spectrum"]
        for sclk in answered.loc[answered["predicted_label"] == c3.LABEL_3OH, "sclk"]:
            with open(os.path.join(plots_dir, f"{sclk}.png"), "wb") as f:
                f.write(c3.render_spectrum_png(spectrum_by_sclk[sclk], f"Sample {sclk}"))

    n_3oh = int((answered["predicted_label"] == c3.LABEL_3OH).sum())
    print(f"Batch done: {len(answered)} answered ({n_3oh} labelled 3-OH), "
          f"{len(batch_meta) - len(answered)} without a usable answer (retried on the next run).")
    os.remove(request_path)

print("\n" + "=" * 50 + "\n3-OH inference run finished.\n" + "=" * 50)


--- Batch 1/30: 1000 spectra (SCLK 1347166080 - 1352959190) ---


Rendering requests:   0%|          | 0/1000 [00:00<?, ?it/s]

Wrote 1000 requests (1356 MB) to /workspaces/cda-dust-analyzer/Notebooks/3oh_classification/inference_results_3oh/batch_requests.jsonl


Uploading batch_requests.jsonl:   0%|          | 0.00/1.36G [00:00<?, ?B/s]

Job Submitted! Name: projects/355771430623/locations/global/batchPredictionJobs/2840160168019230720
[16:00:40] Job State: JOB_STATE_PENDING
[16:06:10] Job State: JOB_STATE_RUNNING
[16:06:42] Job State: JOB_STATE_RUNNING
[16:07:13] Job State: JOB_STATE_RUNNING
[16:07:45] Job State: JOB_STATE_RUNNING
[16:08:17] Job State: JOB_STATE_RUNNING
[16:08:49] Job State: JOB_STATE_RUNNING
[16:25:12] Job State: JOB_STATE_RUNNING
[16:25:44] Job State: JOB_STATE_RUNNING
[16:42:47] Job State: JOB_STATE_SUCCEEDED


...

Uploading batch_requests.jsonl:   0%|          | 0.00/1.36G [00:00<?, ?B/s]

Job Submitted! Name: projects/355771430623/locations/global/batchPredictionJobs/5395812219063631872
[19:23:41] Job State: JOB_STATE_PENDING
[19:24:13] Job State: JOB_STATE_PENDING
[19:24:44] Job State: JOB_STATE_PENDING
[19:25:16] Job State: JOB_STATE_PENDING
[19:25:47] Job State: JOB_STATE_PENDING
[19:26:19] Job State: JOB_STATE_PENDING
[19:26:51] Job State: JOB_STATE_PENDING
[19:27:23] Job State: JOB_STATE_PENDING
[19:27:55] Job State: JOB_STATE_PENDING
[19:28:27] Job State: JOB_STATE_QUEUED
[19:28:58] Job State: JOB_STATE_QUEUED
[19:29:30] Job State: JOB_STATE_RUNNING
[19:30:01] Job State: JOB_STATE_RUNNING
[19:30:33] Job State: JOB_STATE_RUNNING
[19:31:05] Job State: JOB_STATE_RUNNING
[19:31:37] Job State: JOB_STATE_RUNNING
[19:32:08] Job State: JOB_STATE_RUNNING
[19:32:40] Job State: JOB_STATE_RUNNING
[19:33:11] Job State: JOB_STATE_RUNNING
[19:33:42] Job State: JOB_STATE_RUNNING
[19:34:14] Job State: JOB_STATE_RUNNING
[19:34:46] Job State: JOB_STATE_RUNNING
[19:35:18] Job State: 

Batch done: 1000 answered (0 labelled 3-OH), 0 without a usable answer (retried on the next run).

--- Batch 6/30: 1000 spectra (SCLK 1393344760 - 1406338445) ---


Rendering requests:   0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Step 6: Summary and 3-OH Candidates

In [ ]:
if os.path.exists(predictions_path):
    all_predictions = pd.read_csv(predictions_path, dtype={"catalogue_class": str}).drop_duplicates("sclk", keep="last")
    print(f"{len(all_predictions)} spectra classified so far:")
    print(all_predictions["predicted_label"].value_counts().to_string())
    if all_predictions["catalogue_class"].notna().any():
        print("\nPredicted label by catalogue class:")
        print(pd.crosstab(all_predictions["catalogue_class"].fillna("not catalogued"), all_predictions["predicted_label"]))

    candidates = all_predictions[all_predictions["predicted_label"] == c3.LABEL_3OH].sort_values("sclk")
    candidates.to_csv(os.path.join(STATE_DIR, "candidates_3oh.csv"), index=False)
    print(f"\n{len(candidates)} 3-OH candidates written to {os.path.join(STATE_DIR, 'candidates_3oh.csv')}")
    display(candidates.head(20))

    shown = [s for s in candidates["sclk"] if os.path.exists(os.path.join(plots_dir, f"{s}.png"))][:12]
    if shown:
        n_rows = int(np.ceil(len(shown) / 3))
        fig, axes = plt.subplots(n_rows, 3, figsize=(18, 3.2 * n_rows), squeeze=False)
        for ax, sclk in zip(axes.flat, shown):
            ax.imshow(plt.imread(os.path.join(plots_dir, f"{sclk}.png")))
            ax.axis("off")
        for ax in list(axes.flat)[len(shown):]:
            ax.axis("off")
        plt.tight_layout()
        plt.show()
else:
    print("No predictions yet.")